In [ ]:
import sys
sys.path.append('../')
import os.path as osp

import torch
from torch.utils.data import DataLoader
import numpy as np
from tqdm.notebook import tqdm

import qdre
import Camel.equations as equations

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
#DEVICE = 'cpu'
print(DEVICE)

cuda:0


## Look at the data

In [2]:
DATA_DIR = "./data"
SAVE_DIR = "./models"

In [3]:
SAVE = True

source_file = osp.join(DATA_DIR, "base_distribution_mc_data")
target_file = osp.join(DATA_DIR, "target_distribution_mc_data")

if SAVE is True:
    for dataset in ["train", "val", "test"]:
        arr = np.load(source_file + "_" + dataset + ".npy")
        np.save(source_file + "_positives_" + dataset + ".npy",
                arr[arr[:,-1] >= 0])
        np.save(source_file + "_negatives_" + dataset + ".npy",
                arr[arr[:,-1] < 0])
        
        arr = np.load(target_file + "_" + dataset + ".npy")
        np.save(target_file + "_positives_" + dataset + ".npy",
                arr[arr[:,-1] >= 0])
        np.save(target_file + "_negatives_" + dataset + ".npy",
                arr[arr[:,-1] < 0])

In [4]:
source_positive_file = source_file + "_positives"
source_negative_file = source_file + "_negatives"
target_positive_file = target_file + "_positives"
target_negative_file = target_file + "_negatives"

files = [source_positive_file, source_negative_file, target_positive_file, target_negative_file]

combos = [np.array((0,2)), #++
          np.array((0,3)), #+-
          np.array((1,2)), #-+
          np.array((1,3))] #--

train_datasizes = []
val_datasizes = []
test_datasizes = []
for f in files:
    train_datasizes.append(np.load(f + "_train.npy").shape[0])
    val_datasizes.append(np.load(f + "_val.npy").shape[0])
    test_datasizes.append(np.load(f + "_test.npy").shape[0])

train_datasizes = np.array(train_datasizes)
val_datasizes = np.array(val_datasizes)
test_datasizes = np.array(test_datasizes)
train_datasizes

array([1600782,  399218, 1334428,  665572])

## Initiate the datasets

In [5]:
training_settings = [{}, {}, {}, {}]


source_mixture_coef = (4, -1)
source_scales = (2.5, 2.3)
target_mixture_coef = (2, -1)
target_scales = (2, 1.2)

all_mixture_coef = np.array([*source_mixture_coef, *target_mixture_coef])
all_scales = np.array([*source_scales, *target_scales])

train_generator_datas = []
valid_generator_datas = []
test_generator_datas = []

# If we want to use different training sizes for each combo
for i in range(4):
    min_datasizes = (train_datasizes[combos[i]].min(), val_datasizes[combos[i]].min(), test_datasizes[combos[i]].min())
    print(min_datasizes)
    training_settings[i].update({
        "source_file": files[combos[i][0]],
        "target_file": files[combos[i][1]],
        "n_train": int(min_datasizes[0]),
        "n_val": int(min_datasizes[1]),
        "n_test": int(min_datasizes[2]),
        "source_mixture_coef": (1,0),
        "source_scales": (all_scales[combos[i][0]], all_scales[combos[i][0]]),
        "target_mixture_coef": (1,0),
        "target_scales": (all_scales[combos[i][1]], all_scales[combos[i][1]])
    })


    # We flip the labels here so we learn the ratio of Y=0/Y=1 when we use the regular s/(1-s) trick, which is what we want for the subdensities
    train_base_dataset = qdre.preprocessing.Dataset(files[combos[i][0]] + "_train.npy", 0,
                                                     stop_event=training_settings[i]["n_train"])
    valid_base_dataset = qdre.preprocessing.Dataset(files[combos[i][0]] + "_val.npy", 0,
                                                     stop_event=training_settings[i]["n_val"])

    train_target_dataset = qdre.preprocessing.Dataset(files[combos[i][1]] + "_train.npy", 1,
                                                     stop_event=training_settings[i]["n_train"])
    valid_target_dataset = qdre.preprocessing.Dataset(files[combos[i][1]] + "_val.npy", 1,
                                                       stop_event=training_settings[i]["n_val"])


    source_weight_norm = train_base_dataset.process(normalize_weights=True)
    source_valid_weight_norm = valid_base_dataset.process(normalize_weights=True)
    
    target_weight_norm = train_target_dataset.process(normalize_weights=True)
    target_valid_weight_norm = valid_target_dataset.process(normalize_weights=True)

    train_generator_datas.append(qdre.preprocessing.CombinedDataset(train_base_dataset, train_target_dataset))
    valid_generator_datas.append(qdre.preprocessing.CombinedDataset(valid_base_dataset, valid_target_dataset))

(np.int64(1334428), np.int64(400174), np.int64(932634))
(np.int64(665572), np.int64(199826), np.int64(467366))
(np.int64(399218), np.int64(120297), np.int64(279275))
(np.int64(399218), np.int64(120297), np.int64(279275))


## Do some data preprocessing for standardized inputs and weights

In [6]:
X_scalers, train_weight_norms = list(zip(*[qdre.preprocessing.get_scaling(train_generator_data) for train_generator_data in train_generator_datas]))
_, valid_weight_norms = list(zip(*[qdre.preprocessing.get_scaling(valid_generator_data) for valid_generator_data in valid_generator_datas]))
print(train_weight_norms, valid_weight_norms)

100%|██████████| 235/235 [00:00<00:00, 268.40it/s]

(tensor(1.), tensor(1.), tensor(1.), tensor(1.)) (tensor(1.), tensor(1.), tensor(1.), tensor(1.))


## Prepare the data for training

In [7]:
random_seed = 0

torch.manual_seed(random_seed)
batch_size = int(2**8)
[ts.update({
    "batch_size": batch_size,
    "random_seed": random_seed
}) for ts in training_settings]

train_loaders = [DataLoader(train_generator_data, batch_size=batch_size, shuffle=True) for train_generator_data in train_generator_datas]
valid_loaders = [DataLoader(valid_generator_data, batch_size=batch_size, shuffle=False) for valid_generator_data in valid_generator_datas]

## Construct the model

In [8]:
inputs = 2
hidden_nodes = [32, 32]
outputs = 1

models = [qdre.models.Classifier(inputs, hidden_nodes, outputs).to(DEVICE) for _ in range(4)]
print(models)
print("----------")

#model = model.to(DEVICE)

learning_rate = 1e-4
optimizers = [torch.optim.Adam(model.parameters(), lr=learning_rate) for model in models]

# Get the analytical optimal classifier
s_optimals = [equations.optimal_binary_classifier(training_settings[i]["source_mixture_coef"],
                                                  training_settings[i]["source_scales"],
                                                  training_settings[i]["target_mixture_coef"],
                                                  training_settings[i]["target_scales"]) for i in range(4)]

[ts.update({
    "learning_rate": learning_rate,
    "optimizer": "Adam"
}) for ts in training_settings]

[Classifier(
  (model): Sequential(
    (0): Linear(in_features=2, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=1, bias=True)
    (5): Sigmoid()
  )
), Classifier(
  (model): Sequential(
    (0): Linear(in_features=2, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=1, bias=True)
    (5): Sigmoid()
  )
), Classifier(
  (model): Sequential(
    (0): Linear(in_features=2, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=1, bias=True)
    (5): Sigmoid()
  )
), Classifier(
  (model): Sequential(
    (0): Linear(in_features=2, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=32, bias=True)
    (3): ReLU()


[None, None, None, None]

## Perform the training

In [ ]:
for i in range(4):
    modname = "classifier_subdensity_{}_batch{}".format(i+1, batch_size)
    model = models[i]
    optimizer = optimizers[i]
    X_scaler = X_scalers[i]
    train_weight_norm = train_weight_norms[i]
    valid_weight_norm = valid_weight_norms[i]
    train_loader = train_loaders[i]
    valid_loader = valid_loaders[i]
    s_optimal = s_optimals[i]
    
    n_epochs = 1500
    stale_epochs = 0
    best_valid_loss = 99999
    patience = 20
    max_num_batches = int(int(1e5) / batch_size)
    t = tqdm(range(0, n_epochs))
    
    training_losses = [qdre.train.test(
            model,
            train_loader,
            X_scaler=X_scaler,
            weight_norm=train_weight_norm,
            max_num_batches=max_num_batches,
            device=DEVICE,
            progress_bar=False,
            leave=False
        )[0],]
    validation_losses = [qdre.train.test(
            model,
            valid_loader,
            X_scaler=X_scaler,
            weight_norm=valid_weight_norm,
            device=DEVICE,
            progress_bar=False,
            leave=False
        )[0],]
    
    optimal_train_loss = qdre.train.get_optimal_loss(
            s_optimal,
            train_loader,
            weight_norm=train_weight_norm,
            device=DEVICE,
            progress_bar=False
    )
    optimal_valid_loss = qdre.train.get_optimal_loss(
            s_optimal,
            valid_loader,
            weight_norm=valid_weight_norm,
            device=DEVICE,
            progress_bar=False
    )
    training_settings[i].update({
        "optimal_train_loss": optimal_train_loss,
        "optimal_valid_loss": optimal_valid_loss
    })
    
    
    for epoch in t:
        loss = qdre.train.train(
            model,
            optimizer,
            train_loader,
            X_scaler=X_scaler,
            weight_norm=train_weight_norm,
            max_num_batches=max_num_batches,
            device=DEVICE,
            progress_bar=False,
            leave=bool(epoch == n_epochs - 1),
        )
        #loss -= optimal_train_loss
        training_losses.append(loss[0])
    
        valid_loss = qdre.train.test(
            model,
            valid_loader,
            X_scaler=X_scaler,
            weight_norm=valid_weight_norm,
            device=DEVICE,
            progress_bar=False,
            leave=bool(epoch == n_epochs - 1),
        )
        #valid_loss -= optimal_valid_loss
        validation_losses.append(valid_loss[0])
        print("Epoch: {:02d}, Training Loss:   {:.4f}".format(epoch, loss[1]))
        print("           Validation Loss: {:.4f}".format(valid_loss[1]))
    
        if valid_loss[1] < best_valid_loss:
            best_valid_loss = valid_loss[1]
            training_settings[i].update({
                "n_epochs": epoch+1,
                "training_losses": training_losses,
                "validation_losses": validation_losses
            })
            model_metadata = qdre.train.get_model_metadata(training_settings[i], model, X_scaler, train_weight_norm)
            qdre.train.save_model_data(model, model_metadata, savedir=SAVE_DIR, name=modname, save_onnx=False, device=DEVICE)
            print("New best model saved to: {}.zip".format(osp.join(SAVE_DIR, modname)))
            #torch.save(model.state_dict(), modpath)
            stale_epochs = 0
        else:
            print("Stale epoch")
            stale_epochs += 1
        if stale_epochs >= patience:
            print("Early stopping after %i stale epochs" % patience)
            break

  0%|          | 0/1500 [00:00<?, ?it/s]

Epoch: 00, Training Loss:   0.6884
           Validation Loss: 0.6854
New best model saved to: models/classifier_subdensity_1_batch256.zip
Epoch: 01, Training Loss:   0.6790
           Validation Loss: 0.6767
New best model saved to: models/classifier_subdensity_1_batch256.zip
Epoch: 02, Training Loss:   0.6726
           Validation Loss: 0.6721
New best model saved to: models/classifier_subdensity_1_batch256.zip
Epoch: 03, Training Loss:   0.6706
           Validation Loss: 0.6709
New best model saved to: models/classifier_subdensity_1_batch256.zip
Epoch: 04, Training Loss:   0.6690
           Validation Loss: 0.6704
New best model saved to: models/classifier_subdensity_1_batch256.zip
Epoch: 05, Training Loss:   0.6686
           Validation Loss: 0.6702
New best model saved to: models/classifier_subdensity_1_batch256.zip
Epoch: 06, Training Loss:   0.6679
           Validation Loss: 0.6701
New best model saved to: models/classifier_subdensity_1_batch256.zip
Epoch: 07, Training Loss:  

  0%|          | 0/1500 [00:00<?, ?it/s]

Epoch: 00, Training Loss:   0.6659
           Validation Loss: 0.6399
New best model saved to: models/classifier_subdensity_2_batch256.zip
Epoch: 01, Training Loss:   0.6056
           Validation Loss: 0.5707
New best model saved to: models/classifier_subdensity_2_batch256.zip
Epoch: 02, Training Loss:   0.5421
           Validation Loss: 0.5239
New best model saved to: models/classifier_subdensity_2_batch256.zip
Epoch: 03, Training Loss:   0.5146
           Validation Loss: 0.5115
New best model saved to: models/classifier_subdensity_2_batch256.zip
Epoch: 04, Training Loss:   0.5105
           Validation Loss: 0.5090
New best model saved to: models/classifier_subdensity_2_batch256.zip
Epoch: 05, Training Loss:   0.5064
           Validation Loss: 0.5081
New best model saved to: models/classifier_subdensity_2_batch256.zip
Epoch: 06, Training Loss:   0.5064
           Validation Loss: 0.5075
New best model saved to: models/classifier_subdensity_2_batch256.zip
Epoch: 07, Training Loss:  

  0%|          | 0/1500 [00:00<?, ?it/s]

Epoch: 00, Training Loss:   0.6913
           Validation Loss: 0.6901
New best model saved to: models/classifier_subdensity_3_batch256.zip
Epoch: 01, Training Loss:   0.6864
           Validation Loss: 0.6866
New best model saved to: models/classifier_subdensity_3_batch256.zip
Epoch: 02, Training Loss:   0.6840
           Validation Loss: 0.6850
New best model saved to: models/classifier_subdensity_3_batch256.zip
Epoch: 03, Training Loss:   0.6830
           Validation Loss: 0.6845
New best model saved to: models/classifier_subdensity_3_batch256.zip
Epoch: 04, Training Loss:   0.6825
           Validation Loss: 0.6842
New best model saved to: models/classifier_subdensity_3_batch256.zip
Epoch: 05, Training Loss:   0.6831
           Validation Loss: 0.6841
New best model saved to: models/classifier_subdensity_3_batch256.zip
Epoch: 06, Training Loss:   0.6823
           Validation Loss: 0.6840
New best model saved to: models/classifier_subdensity_3_batch256.zip
Epoch: 07, Training Loss:  

  0%|          | 0/1500 [00:00<?, ?it/s]

Epoch: 00, Training Loss:   0.6582
           Validation Loss: 0.6332
New best model saved to: models/classifier_subdensity_4_batch256.zip
Epoch: 01, Training Loss:   0.6037
           Validation Loss: 0.5776
New best model saved to: models/classifier_subdensity_4_batch256.zip
Epoch: 02, Training Loss:   0.5589
           Validation Loss: 0.5466
New best model saved to: models/classifier_subdensity_4_batch256.zip
Epoch: 03, Training Loss:   0.5397
           Validation Loss: 0.5389
New best model saved to: models/classifier_subdensity_4_batch256.zip
Epoch: 04, Training Loss:   0.5383
           Validation Loss: 0.5379
New best model saved to: models/classifier_subdensity_4_batch256.zip
Epoch: 05, Training Loss:   0.5379
           Validation Loss: 0.5376
New best model saved to: models/classifier_subdensity_4_batch256.zip
Epoch: 06, Training Loss:   0.5370
           Validation Loss: 0.5374
New best model saved to: models/classifier_subdensity_4_batch256.zip
Epoch: 07, Training Loss:  